### Test Image Adjust

In [ ]:
# 이미지 한 장 조정 모듈 
"""
입력: 이미지 링크, 저장하고자 하는 위치 

출력: 폴더에 이미지 저장, 

과정
    1. 이미지 링크 가져오기
    2. 링크에서 이미지 로드
    3. 이미지 darkness 정도 파악
    4. darkness 정도에 따라 이미지 조정 파라미터 계산
    5. 계산에 따라 이미지 보정
    6. 보정된 이미지 - 지정된 파일에 저장장

"""

def image_adjust(image_link, save_path):
    pass

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

IMG_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp"}

def get_one_tomatod_3cls_image(split="test", index=0):
    dataset_dir = DATASETS["tomatod_3cls"]
    image_dir = dataset_dir / split / "images"
    images = sorted([p for p in image_dir.iterdir() if p.suffix.lower() in IMG_SUFFIXES])
    return images[index]

def gentle_auto_gamma_bgr(
    image_bgr: np.ndarray,
    target_mean: float = 115.0,
    min_gamma: float = 0.90,
    max_gamma: float = 1.00,
    blend: float = 0.35,
):
    ycrcb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2YCrCb)
    y = ycrcb[:, :, 0]
    mean_in = float(y.mean())

    if mean_in >= target_mean:
        gamma = 1.0
        corrected = image_bgr.copy()
    else:
        mean_norm = max(mean_in / 255.0, 1e-6)
        target_norm = target_mean / 255.0
        gamma = np.log(target_norm) / np.log(mean_norm)
        gamma = float(np.clip(gamma, min_gamma, max_gamma))

        lut = np.array(
            [np.clip(((i / 255.0) ** gamma) * 255.0, 0, 255) for i in range(256)],
            dtype=np.uint8,
        )

        out = ycrcb.copy()
        out[:, :, 0] = cv2.LUT(y, lut)
        corrected = cv2.cvtColor(out, cv2.COLOR_YCrCb2BGR)

    blended = cv2.addWeighted(corrected, blend, image_bgr, 1.0 - blend, 0)

    info = {
        "mean_input": mean_in,
        "gamma": gamma,
        "blend": blend,
        "target_mean": target_mean,
        "mean_output": float(cv2.cvtColor(blended, cv2.COLOR_BGR2YCrCb)[:, :, 0].mean()),
    }
    return blended, info

In [ ]:
def shift_tomato_hue(
    image_bgr: np.ndarray,
    # 타겟 Hue 범위 (주황-빨강 토마토)
    hue_low: int = 0,
    hue_high: int = 25,       # H=0~25 범위를 타겟으로
    hue_shift: int = -8,      # 빨강 방향으로 당기기 (음수=빨강 방향)
    sat_boost: float = 1.4,   # 해당 영역 채도 강화
    min_saturation: int = 40, # 너무 무채색은 제외
    min_value: int = 30,      # 너무 어두운 픽셀 제외
) -> np.ndarray:
    
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV).astype(np.int32)
    h, s, v = hsv[:,:,0], hsv[:,:,1], hsv[:,:,2]
    
    # 토마토 색상 범위 마스크
    mask = (
        ((h >= hue_low) & (h <= hue_high)) &
        (s >= min_saturation) &
        (v >= min_value)
    )
    
    # Hue 시프트 + 채도 강화
    h[mask] = np.clip(h[mask] + hue_shift, 0, 179)
    s[mask] = np.clip(s[mask] * sat_boost, 0, 255)
    
    hsv[:,:,0] = h
    hsv[:,:,1] = s
    result = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    
    print(f"보정된 픽셀 수: {mask.sum()}")
    return result

In [ ]:
def agc_with_selective_color_enhancement(
    image_bgr: np.ndarray,
    expected_intensity: float = 100.0,
    normalize: float = 1.5,
    threshold: float = 0.3,
    sat_scale_low: float = 2.5,
    sat_scale_high: float = 1.1,
    sat_threshold: float = 60.0,
    lab_ab_scale: float = 1.3,
    blend: float = 0.75,
    # ── 추가: 밝기 마스킹 ──────────────────
    brightness_mask: bool = True,       # 어두운 영역만 보정
    bright_pixel_threshold: int = 80,   # 이 값 이상이면 "이미 밝음" → 보정 약하게
    bright_pixel_blend: float = 0.15,   # 밝은 영역에 보정 반영 비율
) -> tuple[np.ndarray, dict]:

    # ── 1. AGC 밝기 보정 ─────────────────────────────────────────────
    ycrcb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2YCrCb)
    y = ycrcb[:, :, 0].astype(np.uint8)
    mean_in = float(y.mean())
    t = (mean_in - expected_intensity) / expected_intensity

    if abs(t) <= threshold:
        brightness_corrected = image_bgr.copy()
    else:
        reversed_mode = t > threshold
        work_y = 255 - y if reversed_mode else y

        hist = cv2.calcHist([work_y], [0], None, [256], [0, 256]).flatten()
        prob = hist / hist.sum()
        prob_min, prob_max = prob.min(), prob.max()

        pn_temp = np.zeros_like(prob, dtype=np.float64)
        mask = prob > 0
        pn_temp[mask] = ((prob[mask] - prob_min) / (prob_max - prob_min)) ** normalize
        prob_wd = pn_temp / pn_temp.sum() * normalize
        cdf_wd = np.cumsum(prob_wd)
        inverse_cdf = normalize - cdf_wd

        lut = np.array([
            np.clip(round(255 * ((i / 255.0) ** inverse_cdf[i])), 0, 255)
            for i in range(256)
        ], dtype=np.uint8)

        corrected_y = lut[work_y]
        if reversed_mode:
            corrected_y = 255 - corrected_y

        if brightness_mask:
            # 밝은 픽셀은 보정을 거의 안 받도록 픽셀별 블렌딩
            dark_mask = (y < bright_pixel_threshold).astype(np.float32)  # 0 or 1
            # 어두운 픽셀: 100% 보정 / 밝은 픽셀: bright_pixel_blend만 보정
            blend_map = dark_mask + (1 - dark_mask) * bright_pixel_blend
            corrected_y = (corrected_y * blend_map + y * (1 - blend_map)).astype(np.uint8)

        out = ycrcb.copy()
        out[:, :, 0] = corrected_y
        brightness_corrected = cv2.cvtColor(out, cv2.COLOR_YCrCb2BGR)

    # ── 2. 채도 선택적 강화 ───────────────────────────────────────────
    hsv = cv2.cvtColor(brightness_corrected, cv2.COLOR_BGR2HSV).astype(np.float32)
    s = hsv[:, :, 1]
    scale_map = np.where(s < sat_threshold, sat_scale_low, sat_scale_high)
    hsv[:, :, 1] = np.clip(s * scale_map, 0, 255)
    sat_enhanced = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

    # ── 3. LAB 색 대비 강화 ───────────────────────────────────────────
    lab = cv2.cvtColor(sat_enhanced, cv2.COLOR_BGR2LAB).astype(np.float32)
    
    # a채널(빨강-초록): 128 중심으로 스케일 → 빨강은 더 빨갛게, 초록은 더 초록으로
    a = lab[:, :, 1]
    lab[:, :, 1] = np.clip((a - 128) * lab_ab_scale + 128, 0, 255)
    
    # b채널(노랑-파랑): 동일하게 128 중심
    b = lab[:, :, 2]
    lab[:, :, 2] = np.clip((b - 128) * lab_ab_scale + 128, 0, 255)
    
    color_enhanced = cv2.cvtColor(lab.astype(np.uint8), cv2.COLOR_LAB2BGR)
    
    # ── 3.5 토마토 Hue 보정 ──────────────────────────────────────────
    
    color_enhanced = shift_tomato_hue(
        color_enhanced,
        hue_low=0,
        hue_high=50,
        hue_shift=-10,
        sat_boost=1.5,
        min_saturation=85,   # 35 → 80 (흙/갈색 제외)
        min_value=60,        # 25 → 60 (너무 어둡거나 탁한 픽셀 제외)
    )
    
    # ── 4. 블렌딩 ─────────────────────────────────────────────────────
    result = cv2.addWeighted(color_enhanced, blend, image_bgr, 1.0 - blend, 0)

    info = {
        "mean_input": mean_in,
        "mean_output": float(cv2.cvtColor(result, cv2.COLOR_BGR2YCrCb)[:, :, 0].mean()),
        "sat_scale_low": sat_scale_low,
        "sat_scale_high": sat_scale_high,
        "sat_threshold": sat_threshold,
        "lab_ab_scale": lab_ab_scale,
        "blend": blend,
        "bright_pixel_threshold": bright_pixel_threshold,
        "bright_pixel_blend": bright_pixel_blend,
    }
    return result, info

In [ ]:
def agc_tomato_focused(
    image_bgr: np.ndarray,
    # AGC
    expected_intensity: float = 90.0,
    normalize: float = 1.5,
    threshold: float = 0.3,
    bright_pixel_threshold: int = 80,
    bright_pixel_blend: float = 0.15,
    # 토마토 색상 타겟 보정만
    tomato_hue_low: int = 0,
    tomato_hue_high: int = 50,
    tomato_sat_boost: float = 1.8,
    tomato_min_sat: int = 80,
    tomato_min_val: int = 60,
    tomato_hue_shift: int = -5,
    # 블렌딩
    blend: float = 0.65,
) -> tuple[np.ndarray, dict]:

    # ── 1. AGC 밝기 보정 ─────────────────────────────────────────────
    ycrcb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2YCrCb)
    y = ycrcb[:, :, 0].astype(np.uint8)
    mean_in = float(y.mean())
    t = (mean_in - expected_intensity) / expected_intensity

    if abs(t) <= threshold:
        brightness_corrected = image_bgr.copy()
    else:
        reversed_mode = t > threshold
        work_y = 255 - y if reversed_mode else y

        hist = cv2.calcHist([work_y], [0], None, [256], [0, 256]).flatten()
        prob = hist / hist.sum()
        prob_min, prob_max = prob.min(), prob.max()
        pn_temp = np.zeros_like(prob, dtype=np.float64)
        mask = prob > 0
        pn_temp[mask] = ((prob[mask] - prob_min) / (prob_max - prob_min)) ** normalize
        prob_wd = pn_temp / pn_temp.sum() * normalize
        inverse_cdf = normalize - np.cumsum(prob_wd)

        lut = np.array([
            np.clip(round(255 * ((i / 255.0) ** inverse_cdf[i])), 0, 255)
            for i in range(256)
        ], dtype=np.uint8)

        corrected_y = lut[work_y]
        if reversed_mode:
            corrected_y = 255 - corrected_y

        # 밝은 픽셀은 보정 최소화
        dark_mask = (y < bright_pixel_threshold).astype(np.float32)
        blend_map = dark_mask + (1 - dark_mask) * bright_pixel_blend
        corrected_y = (corrected_y * blend_map + y * (1 - blend_map)).astype(np.uint8)

        out = ycrcb.copy()
        out[:, :, 0] = corrected_y
        brightness_corrected = cv2.cvtColor(out, cv2.COLOR_YCrCb2BGR)

    # ── 2. 토마토 Hue 타겟 보정만 ────────────────────────────────────
    hsv = cv2.cvtColor(brightness_corrected, cv2.COLOR_BGR2HSV).astype(np.int32)
    h, s, v = hsv[:,:,0], hsv[:,:,1], hsv[:,:,2]

    tomato_mask = (
        (h >= tomato_hue_low) & (h <= tomato_hue_high) &
        (s >= tomato_min_sat) &
        (v >= tomato_min_val)
    )
    h[tomato_mask] = np.clip(h[tomato_mask] + tomato_hue_shift, 0, 179)
    s[tomato_mask] = np.clip(s[tomato_mask] * tomato_sat_boost, 0, 255)

    hsv[:,:,0] = h
    hsv[:,:,1] = s
    color_corrected = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

    # ── 3. 블렌딩 ─────────────────────────────────────────────────────
    result = cv2.addWeighted(color_corrected, blend, image_bgr, 1.0 - blend, 0)

    info = {
        "mean_input": mean_in,
        "mean_output": float(cv2.cvtColor(result, cv2.COLOR_BGR2YCrCb)[:, :, 0].mean()),
        "tomato_pixels_corrected": int(tomato_mask.sum()),
        "tomato_sat_boost": tomato_sat_boost,
        "tomato_hue_shift": tomato_hue_shift,
        "blend": blend,
    }
    return result, info

In [ ]:
img_path = get_one_tomatod_3cls_image(split="test", index=30)
img_bgr = cv2.imread(str(img_path))

result_bgr, info = agc_with_selective_color_enhancement(
    img_bgr,
    expected_intensity=100.0,
    sat_scale_low=2.0,
    sat_scale_high=1.0,
    sat_threshold=60.0,
    lab_ab_scale=1.2,
    blend=0.7,
)

print("image:", img_path.name)
for k, v in info.items():
    print(f"{k}: {v}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB))
axes[1].set_title(
    f"AGC + Selective Color Enhanced\n"
    f"mean {info['mean_input']:.1f} → {info['mean_output']:.1f} | "
    f"sat_low×{info['sat_scale_low']} sat_high×{info['sat_scale_high']} | "
    f"lab_ab×{info['lab_ab_scale']}"
)
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
result_bgr, info = agc_tomato_focused(img_bgr)

print("image:", img_path.name)
for k, v in info.items():
    print(f"{k}: {v}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB))
axes[1].set_title(
    f"AGC + Tomato Focused\n"
    f"mean {info['mean_input']:.1f} → {info['mean_output']:.1f} | "
    f"tomato_px={info['tomato_pixels_corrected']} | "
    f"sat×{info['tomato_sat_boost']}"
)
axes[1].axis("off")

plt.tight_layout()
plt.show()